# 🧹 Data Cleaning & Formatting – Strava Activities

This notebook prepares the raw data collected from Strava by cleaning, transforming, and formatting it for analysis.

The cleaned data is saved as a separate CSV file for use in reports, visualizations, and summaries.

---


## ✅ What this notebook does

1. Loads the collected raw activity data (`atividades_todos.csv`)
2. Filters only cycling activities (`Ride`)
3. Extracts year and month
4. Converts time and speed units
5. Selects and renames final columns
6. Saves the formatted result as `atividades_formatado.csv`


### 1. 📦 Load Required Libraries

In [1]:
import os
import pandas as pd
from dotenv import load_dotenv

### 2. ⚙️ Load Environment Variables

In [2]:
# Load .env variables
load_dotenv()

ACTIVITIES_PATH = os.getenv("ACTIVITIES_PATH")
FORMATTED_PATH = os.getenv("FORMATTED_PATH")

### 3. 🔁 Load raw activity data

In [3]:
# Load the raw activity data
df = pd.read_csv(ACTIVITIES_PATH)
df['start_date_local'] = pd.to_datetime(df['start_date_local'])

# Filter only Ride activities
# df = df[df['type'] == 'Ride'].copy()

In [4]:
mask_ride = df['type'] == 'Ride'
mask_cycle = df['type'] == 'Cycle'
df_ride = df[mask_ride].copy()
df_cycle = df[mask_cycle].copy()

### 4. 🧹 Clean and Process Data

In [5]:
# # Convert distance to km
# df['distance'] = (df['distance'] / 1000).round(2)

# # Extract date components
# df['date'] = pd.to_datetime(df['start_date_local']).dt.date

# # Convert time columns to hh:mm:ss format
# df['moving_time'] = pd.to_timedelta(df['moving_time'], unit='s')
# df['elapsed_time'] = pd.to_timedelta(df['elapsed_time'], unit='s')
# df['moving_time'] = df['moving_time'].apply(lambda x: str(x).split()[-1])
# df['elapsed_time'] = df['elapsed_time'].apply(lambda x: str(x).split()[-1])

# # Convert speed to km/h
# df['average_speed'] = (df['average_speed'] * 3.6).round(1)
# df['max_speed'] = (df['max_speed'] * 3.6).round(1)

# # Heart rate
# df['average_heartrate'] = pd.to_numeric(df.get('average_heartrate', None), errors='coerce')
# df['max_heartrate'] = pd.to_numeric(df.get('max_heartrate', None), errors='coerce')

# # Rename the column "nome" to "athlete"
# df.rename(columns={'nome': 'athlete'}, inplace=True)

In [5]:
# ----------- Ride pipeline -----------
if not df_ride.empty:
    df_ride['distance'] = (df_ride['distance'] / 1000).round(2)
    df_ride['date'] = pd.to_datetime(df_ride['start_date_local']).dt.date
    df_ride['moving_time'] = pd.to_timedelta(df_ride['moving_time'], unit='s')
    df_ride['elapsed_time'] = pd.to_timedelta(df_ride['elapsed_time'], unit='s')
    df_ride['moving_time'] = df_ride['moving_time'].apply(lambda x: str(x).split()[-1])
    df_ride['elapsed_time'] = df_ride['elapsed_time'].apply(lambda x: str(x).split()[-1])
    df_ride['average_speed'] = (df_ride['average_speed'] * 3.6).round(1)
    df_ride['max_speed'] = (df_ride['max_speed'] * 3.6).round(1)
    df_ride['average_heartrate'] = pd.to_numeric(df_ride.get('average_heartrate', None), errors='coerce')
    df_ride['max_heartrate'] = pd.to_numeric(df_ride.get('max_heartrate', None), errors='coerce')
    df_ride.rename(columns={'nome': 'athlete'}, inplace=True)

# ----------- Cycle pipeline (apenas tempo) -----------
if not df_cycle.empty:
    df_cycle['date'] = pd.to_datetime(df_cycle['start_date_local']).dt.date
    df_cycle['moving_time'] = pd.to_timedelta(df_cycle['moving_time'], unit='s')
    df_cycle['moving_time'] = df_cycle['moving_time'].apply(lambda x: str(x).split()[-1])
    df_cycle.rename(columns={'nome': 'athlete'}, inplace=True)

# Colunas finais
cols_ride = ['athlete', 'date', 'distance', 'moving_time', 'elapsed_time', 'average_speed', 'max_speed', 'average_heartrate', 'max_heartrate']
cols_cycle = ['athlete', 'date', 'moving_time']

# Garantir que só seleciona colunas presentes
cols_ride = [col for col in cols_ride if col in df_ride.columns]
cols_cycle = [col for col in cols_cycle if col in df_cycle.columns]

# Dataframes separados
ride_out = df_ride[cols_ride].copy() if not df_ride.empty else pd.DataFrame()
cycle_out = df_cycle[cols_cycle].copy() if not df_cycle.empty else pd.DataFrame()

# Identificar tipo de atividade
ride_out['activity_type'] = 'Ride'
cycle_out['activity_type'] = 'Cycle'


In [6]:
# df.head()

# Concatenar
combined = pd.concat([ride_out, cycle_out], ignore_index=True, sort=True)

# Organizar ordem das colunas sugeridas
final_cols = ['athlete', 'activity_type', 'date', 'distance', 'moving_time', 'elapsed_time', 'average_speed', 'max_speed', 'average_heartrate', 'max_heartrate']
final_cols = [col for col in final_cols if col in combined.columns]
df_formatted = combined[final_cols].copy()

## 5. 🧾 Select and rename final columns 

In [6]:
# df_formatted = df[[
#     'athlete', 'date', 'distance', 'moving_time',
#     'elapsed_time', 'average_speed', 'max_speed',
#     'average_heartrate', 'max_heartrate']].copy()

In [7]:
df_formatted.head()

,athlete,activity_type,date,distance,moving_time,elapsed_time,average_speed,max_speed,average_heartrate,max_heartrate
0,Diego Galdino,Ride,2025-08-31,151.66,07:52:33,09:36:09,19.3,48.7,125.2,161.0
1,Diego Galdino,Ride,2025-08-24,40.03,01:41:04,01:45:33,23.8,37.0,NaN,NaN
2,Diego Galdino,Ride,2025-08-10,111.23,05:45:21,07:04:30,19.3,30.6,NaN,NaN
3,Diego Galdino,Ride,2025-08-03,50.27,01:59:42,02:04:44,25.2,36.1,NaN,NaN
4,Diego Galdino,Ride,2025-07-28,17.05,00:55:35,00:55:35,18.4,39.4,109.3,143.0


### 5. 📥Save formatted file

In [8]:
df_formatted.to_csv(FORMATTED_PATH, index=False)
print(f"✅ Formatted data saved to: {FORMATTED_PATH}")


✅ Formatted data saved to: C:/Users/dsgal/OneDrive/Documentos/Data_Analysis/Flat&Furious/data/activities_formatted.csv
